## Setup and hypothesis definition

In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import kruskal, chi2_contingency
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sp

PROJECT_ROOT = Path(r"F:\StressGNN")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_DEV_PATH = PROCESSED_DIR / "train_dev.csv"
RESULTS_PATH = PROCESSED_DIR / "hypothesis_test_results.csv"
DESCRIPTIVE_PATH = PROCESSED_DIR / "hypothesis_descriptive_statistics.csv"

TARGET = "Stress_Detection"
STRESS_ORDER = ["Low", "Medium", "High"]

NUMERICAL_HYPOTHESES = [
    "Sleep_Duration",
    "Physical_Activity",
    "Screen_Time",
    "Work_Hours"
]

CATEGORICAL_HYPOTHESES = [
    "Gender",
    "Marital_Status",
    "Smoking_Habit",
    "Meditation_Practice",
    "Exercise_Type"
]

ALPHA = 0.05

df = pd.read_csv(TRAIN_DEV_PATH)

assert TARGET in df.columns
assert df[TARGET].notna().all()
assert not df.duplicated().any()

df[TARGET] = pd.Categorical(
    df[TARGET],
    categories=STRESS_ORDER,
    ordered=True
)

print("Dataset shape:", df.shape)
print("Target:", TARGET)
print("Numerical hypotheses:", NUMERICAL_HYPOTHESES)
print("Categorical hypotheses:", CATEGORICAL_HYPOTHESES)
print("Total hypotheses:", len(NUMERICAL_HYPOTHESES) + len(CATEGORICAL_HYPOTHESES))
print("Frozen test set used:", False)

Dataset shape: (2400, 22)
Target: Stress_Detection
Numerical hypotheses: ['Sleep_Duration', 'Physical_Activity', 'Screen_Time', 'Work_Hours']
Categorical hypotheses: ['Gender', 'Marital_Status', 'Smoking_Habit', 'Meditation_Practice', 'Exercise_Type']
Total hypotheses: 9
Frozen test set used: False


## Descriptive statistics by stress class

In [21]:
descriptive_rows = []

for feature in NUMERICAL_HYPOTHESES:
    for stress_level in STRESS_ORDER:
        values = df.loc[
            df[TARGET] == stress_level,
            feature
        ].dropna()

        descriptive_rows.append({
            "Feature": feature,
            "Stress_Level": stress_level,
            "N": len(values),
            "Mean": values.mean(),
            "Median": values.median(),
            "Std": values.std(),
            "Min": values.min(),
            "Max": values.max()
        })

descriptive_statistics = pd.DataFrame(descriptive_rows)

display(
    descriptive_statistics.style.format({
        "Mean": "{:.3f}",
        "Median": "{:.3f}",
        "Std": "{:.3f}",
        "Min": "{:.3f}",
        "Max": "{:.3f}"
    })
)

,Feature,Stress_Level,N,Mean,Median,Std,Min,Max
0,Sleep_Duration,Low,499,6.616,6.720,1.191,2.250,10.020
1,Sleep_Duration,Medium,1007,6.309,6.300,1.113,2.400,9.630
2,Sleep_Duration,High,894,6.132,6.190,1.101,1.920,9.720
3,Physical_Activity,Low,499,2.562,2.610,1.083,-0.620,6.120
4,Physical_Activity,Medium,1007,2.752,3.000,1.125,-1.340,6.560
5,Physical_Activity,High,894,3.376,3.480,1.147,-1.100,6.410
6,Screen_Time,Low,499,3.487,3.410,1.186,-0.440,6.850
7,Screen_Time,Medium,1007,3.883,4.000,1.083,0.160,8.640
8,Screen_Time,High,894,4.570,4.860,1.079,0.820,8.200
9,Work_Hours,Low,499,8.054,8.000,1.283,3.280,12.530


## Kruskal-Wallis Tests for Numerical Features

In [22]:
def epsilon_squared_kw(h_statistic, n_groups, total_n):
    return max(
        0,
        (h_statistic - n_groups + 1) / (total_n - n_groups)
    )

def format_p_value(p):
    if p < 0.001:
        return "< 0.001"
    return f"{p:.3f}"

numerical_results = []

for feature in NUMERICAL_HYPOTHESES:
    groups = [
        df.loc[df[TARGET] == stress_level, feature].dropna()
        for stress_level in STRESS_ORDER
    ]

    h_statistic, p_value = kruskal(*groups)

    total_n = sum(len(group) for group in groups)
    n_groups = len(groups)

    epsilon_sq = epsilon_squared_kw(
        h_statistic,
        n_groups,
        total_n
    )

    numerical_results.append({
        "Feature": feature,
        "Test_Type": "Kruskal-Wallis",
        "Statistic": h_statistic,
        "p_value": p_value,
        "Effect_Size": epsilon_sq,
        "Effect_Size_Type": "Epsilon-squared",
        "p_value_reported": format_p_value(p_value)
    })

numerical_results = pd.DataFrame(numerical_results)

display(
    numerical_results[
        [
            "Feature",
            "Test_Type",
            "Statistic",
            "p_value_reported",
            "Effect_Size",
            "Effect_Size_Type"
        ]
    ].style.format({
        "Statistic": "{:.5f}",
        "Effect_Size": "{:.5f}"
    })
)

,Feature,Test_Type,Statistic,p_value_reported,Effect_Size,Effect_Size_Type
0,Sleep_Duration,Kruskal-Wallis,78.23642,< 0.001,0.03180,Epsilon-squared
1,Physical_Activity,Kruskal-Wallis,216.28338,< 0.001,0.08940,Epsilon-squared
2,Screen_Time,Kruskal-Wallis,328.03671,< 0.001,0.13602,Epsilon-squared
3,Work_Hours,Kruskal-Wallis,139.86046,< 0.001,0.05751,Epsilon-squared


## Categorical Chi-square tests

In [23]:
def cramers_v(chi2_stat, n, r, k):
    denominator = n * min(r - 1, k - 1)
    if denominator == 0:
        return np.nan
    return np.sqrt(chi2_stat / denominator)

categorical_results = []

for feature in CATEGORICAL_HYPOTHESES:
    contingency_table = pd.crosstab(
        df[feature],
        df[TARGET]
    )

    contingency_table = contingency_table.reindex(
        columns=STRESS_ORDER,
        fill_value=0
    )

    chi2_stat, p_value, degrees_freedom, expected = chi2_contingency(
        contingency_table
    )

    n = contingency_table.to_numpy().sum()
    rows, columns = contingency_table.shape

    v = cramers_v(
        chi2_stat,
        n,
        rows,
        columns
    )

    expected_cells_below_5 = int((expected < 5).sum())
    expected_cells_percentage = (
        expected_cells_below_5 / expected.size
    ) * 100

    categorical_results.append({
        "Feature": feature,
        "Test_Type": "Chi-Square",
        "Statistic": chi2_stat,
        "Degrees_of_Freedom": degrees_freedom,
        "p_value": p_value,
        "Effect_Size": v,
        "Effect_Size_Type": "Cramer's V",
        "Min_Expected": expected.min(),
        "Expected_Cells_<5": expected_cells_below_5,
        "Expected_Cells_<5_%": expected_cells_percentage,
        "p_value_reported": format_p_value(p_value)
    })

categorical_results = pd.DataFrame(categorical_results)

display(
    categorical_results[
        [
            "Feature",
            "Test_Type",
            "Statistic",
            "Degrees_of_Freedom",
            "p_value_reported",
            "Effect_Size",
            "Min_Expected",
            "Expected_Cells_<5",
            "Expected_Cells_<5_%"
        ]
    ].style.format({
        "Statistic": "{:.5f}",
        "Effect_Size": "{:.5f}",
        "Min_Expected": "{:.5f}",
        "Expected_Cells_<5_%": "{:.2f}"
    })
)

,Feature,Test_Type,Statistic,Degrees_of_Freedom,p_value_reported,Effect_Size,Min_Expected,Expected_Cells_<5,Expected_Cells_<5_%
0,Gender,Chi-Square,51.16058,2,< 0.001,0.14600,246.38125,0,0.00
1,Marital_Status,Chi-Square,204.54425,4,< 0.001,0.20643,44.28625,0,0.00
2,Smoking_Habit,Chi-Square,211.05832,2,< 0.001,0.29655,233.49042,0,0.00
3,Meditation_Practice,Chi-Square,221.84902,2,< 0.001,0.30403,193.77833,0,0.00
4,Exercise_Type,Chi-Square,191.10139,12,< 0.001,0.19953,2.70292,4,19.05


In [24]:
exercise_result = categorical_results[
    categorical_results["Feature"] == "Exercise_Type"
].iloc[0]

if exercise_result["Expected_Cells_<5"] > 0:
    print("Warning: Exercise_Type has expected-frequency cells below 5.")
    print(
        f"Cells below 5: {int(exercise_result['Expected_Cells_<5'])}"
    )
    print(
        f"Percentage: {exercise_result['Expected_Cells_<5_%']:.2f}%"
    )
else:
    print("Exercise_Type satisfies the expected-frequency diagnostic.")

Cells below 5: 4
Percentage: 19.05%


## Holm multiple-testing correction

In [25]:
numerical_holm = numerical_results[
    [
        "Feature",
        "Test_Type",
        "Statistic",
        "p_value",
        "Effect_Size",
        "Effect_Size_Type"
    ]
].copy()

categorical_holm = categorical_results[
    [
        "Feature",
        "Test_Type",
        "Statistic",
        "p_value",
        "Effect_Size",
        "Effect_Size_Type"
    ]
].copy()

combined_results = pd.concat(
    [
        numerical_holm,
        categorical_holm
    ],
    ignore_index=True
)

reject, p_values_holm, _, _ = multipletests(
    combined_results["p_value"],
    alpha=ALPHA,
    method="holm"
)

combined_results["p_value_Holm"] = p_values_holm
combined_results["Significant_After_Holm"] = reject
combined_results["p_Holm_reported"] = combined_results[
    "p_value_Holm"
].apply(format_p_value)

def effect_interpretation(value, effect_type):
    if effect_type == "Epsilon-squared":
        if value < 0.01:
            return "Negligible"
        elif value < 0.06:
            return "Small"
        elif value < 0.14:
            return "Medium"
        return "Large"

    if effect_type == "Cramer's V":
        if value < 0.10:
            return "Negligible"
        elif value < 0.30:
            return "Small"
        elif value < 0.50:
            return "Medium"
        return "Large"

    return "Not classified"

combined_results["Effect_Interpretation"] = combined_results.apply(
    lambda row: effect_interpretation(
        row["Effect_Size"],
        row["Effect_Size_Type"]
    ),
    axis=1
)

display(
    combined_results[
        [
            "Feature",
            "Test_Type",
            "Statistic",
            "p_value_Holm",
            "p_Holm_reported",
            "Effect_Size",
            "Effect_Size_Type",
            "Effect_Interpretation",
            "Significant_After_Holm"
        ]
    ].sort_values(
        "p_value_Holm"
    ).style.format({
        "Statistic": "{:.5f}",
        "p_value_Holm": "{:.6g}",
        "Effect_Size": "{:.5f}"
    })
)

,Feature,Test_Type,Statistic,p_value_Holm,p_Holm_reported,Effect_Size,Effect_Size_Type,Effect_Interpretation,Significant_After_Holm
2,Screen_Time,Kruskal-Wallis,328.03671,5.27201e-71,< 0.001,0.13602,Epsilon-squared,Medium,True
7,Meditation_Practice,Chi-Square,221.84902,5.36027e-48,< 0.001,0.30403,Cramer's V,Medium,True
1,Physical_Activity,Kruskal-Wallis,216.28338,7.58158e-47,< 0.001,0.08940,Epsilon-squared,Medium,True
6,Smoking_Habit,Chi-Square,211.05832,8.85969e-46,< 0.001,0.29655,Cramer's V,Small,True
5,Marital_Status,Chi-Square,204.54425,1.98031e-42,< 0.001,0.20643,Cramer's V,Small,True
8,Exercise_Type,Chi-Square,191.10139,8.91232e-34,< 0.001,0.19953,Cramer's V,Small,True
3,Work_Hours,Kruskal-Wallis,139.86046,1.27882e-30,< 0.001,0.05751,Epsilon-squared,Small,True
0,Sleep_Duration,Kruskal-Wallis,78.23642,2.05214e-17,< 0.001,0.03180,Epsilon-squared,Small,True
4,Gender,Chi-Square,51.16058,7.7736e-12,< 0.001,0.14600,Cramer's V,Small,True


## Final Hypothesis Testing Results

In [27]:
final_results = combined_results.copy()

final_results["Hypothesis"] = np.where(
    final_results["Test_Type"] == "Kruskal-Wallis",
    "Stress level differs across " + final_results["Feature"],
    final_results["Feature"] + " is associated with stress level"
)

final_results["p_value_reported"] = final_results[
    "p_value"
].apply(format_p_value)

final_results["p_Holm_reported"] = final_results[
    "p_value_Holm"
].apply(format_p_value)

final_results = final_results[
    [
        "Feature",
        "Hypothesis",
        "Test_Type",
        "Statistic",
        "p_value",
        "p_value_reported",
        "p_value_Holm",
        "p_Holm_reported",
        "Effect_Size",
        "Effect_Size_Type",
        "Effect_Interpretation",
        "Significant_After_Holm"
    ]
]

display(
    final_results.style.format({
        "Statistic": "{:.5f}",
        "p_value": "{:.6g}",
        "p_value_Holm": "{:.6g}",
        "Effect_Size": "{:.5f}"
    })
)

,Feature,Hypothesis,Test_Type,Statistic,p_value,p_value_reported,p_value_Holm,p_Holm_reported,Effect_Size,Effect_Size_Type,Effect_Interpretation,Significant_After_Holm
0,Sleep_Duration,Stress level differs across Sleep_Duration,Kruskal-Wallis,78.23642,1.02607e-17,< 0.001,2.05214e-17,< 0.001,0.03180,Epsilon-squared,Small,True
1,Physical_Activity,Stress level differs across Physical_Activity,Kruskal-Wallis,216.28338,1.08308e-47,< 0.001,7.58158e-47,< 0.001,0.08940,Epsilon-squared,Medium,True
2,Screen_Time,Stress level differs across Screen_Time,Kruskal-Wallis,328.03671,5.85779e-72,< 0.001,5.27201e-71,< 0.001,0.13602,Epsilon-squared,Medium,True
3,Work_Hours,Stress level differs across Work_Hours,Kruskal-Wallis,139.86046,4.26273e-31,< 0.001,1.27882e-30,< 0.001,0.05751,Epsilon-squared,Small,True
4,Gender,Gender is associated with stress level,Chi-Square,51.16058,7.7736e-12,< 0.001,7.7736e-12,< 0.001,0.14600,Cramer's V,Small,True
5,Marital_Status,Marital_Status is associated with stress level,Chi-Square,204.54425,3.96062e-43,< 0.001,1.98031e-42,< 0.001,0.20643,Cramer's V,Small,True
6,Smoking_Habit,Smoking_Habit is associated with stress level,Chi-Square,211.05832,1.47661e-46,< 0.001,8.85969e-46,< 0.001,0.29655,Cramer's V,Small,True
7,Meditation_Practice,Meditation_Practice is associated with stress level,Chi-Square,221.84902,6.70034e-49,< 0.001,5.36027e-48,< 0.001,0.30403,Cramer's V,Medium,True
8,Exercise_Type,Exercise_Type is associated with stress level,Chi-Square,191.10139,2.22808e-34,< 0.001,8.91232e-34,< 0.001,0.19953,Cramer's V,Small,True


## Dunn post-hoc tests

Only significant numerical variables after Holm correction get post-hoc testing.

In [31]:
significant_numerical_features = combined_results.loc[
    (combined_results["Test_Type"] == "Kruskal-Wallis") &
    (combined_results["Significant_After_Holm"]),
    "Feature"
].tolist()

dunn_results = {}

for feature in significant_numerical_features:
    posthoc_data = df[[TARGET, feature]].dropna().copy()

    posthoc_data[TARGET] = pd.Categorical(
        posthoc_data[TARGET],
        categories=STRESS_ORDER,
        ordered=True
    )

    dunn_table = sp.posthoc_dunn(
        posthoc_data,
        val_col=feature,
        group_col=TARGET,
        p_adjust="holm"
    )

    dunn_table = dunn_table.reindex(
        index=STRESS_ORDER,
        columns=STRESS_ORDER
    )

    dunn_results[feature] = dunn_table

    print(f"\n{feature}")
    display(
        dunn_table.style.format(
            lambda x: "< 0.001" if x < 0.001 else f"{x:.4f}"
        )
    )


Sleep_Duration


,Low,Medium,High
Low,1.0000,< 0.001,< 0.001
Medium,< 0.001,1.0000,< 0.001
High,< 0.001,< 0.001,1.0000



Physical_Activity


,Low,Medium,High
Low,1.0000,0.0017,< 0.001
Medium,0.0017,1.0000,< 0.001
High,< 0.001,< 0.001,1.0000



Screen_Time


,Low,Medium,High
Low,1.0000,< 0.001,< 0.001
Medium,< 0.001,1.0000,< 0.001
High,< 0.001,< 0.001,1.0000



Work_Hours


,Low,Medium,High
Low,1.0000,0.2967,< 0.001
Medium,0.2967,1.0000,< 0.001
High,< 0.001,< 0.001,1.0000


## Save Hypothesis Testing Results

In [29]:
save_results = combined_results.copy()

save_results.to_csv(
    RESULTS_PATH,
    index=False
)

descriptive_statistics.to_csv(
    DESCRIPTIVE_PATH,
    index=False
)

print("Saved:")
print(RESULTS_PATH)
print(DESCRIPTIVE_PATH)

Saved:
F:\StressGNN\data\processed\hypothesis_test_results.csv
F:\StressGNN\data\processed\hypothesis_descriptive_statistics.csv


## Final Summary 

In [30]:
print("FINAL HYPOTHESIS TESTING SUMMARY")


print(f"Dataset used: train_dev.csv")
print(f"Observations: {len(df)}")
print(f"Primary hypotheses tested: {len(combined_results)}")
print(
    f"Significant after Holm correction: "
    f"{combined_results['Significant_After_Holm'].sum()}"
)

print("\nNumerical hypotheses:")
for _, row in combined_results[
    combined_results["Test_Type"] == "Kruskal-Wallis"
].sort_values("Effect_Size", ascending=False).iterrows():
    print(
        f"{row['Feature']}: "
        f"p {row['p_Holm_reported']}, "
        f"epsilon² = {row['Effect_Size']:.4f}"
    )

print("\nCategorical hypotheses:")
for _, row in combined_results[
    combined_results["Test_Type"] == "Chi-Square"
].sort_values("Effect_Size", ascending=False).iterrows():
    print(
        f"{row['Feature']}: "
        f"p {row['p_Holm_reported']}, "
        f"Cramer's V = {row['Effect_Size']:.4f}"
    )

print("\nFrozen test set used:", False)

FINAL HYPOTHESIS TESTING SUMMARY
Dataset used: train_dev.csv
Observations: 2400
Primary hypotheses tested: 9
Significant after Holm correction: 9

Numerical hypotheses:
Screen_Time: p < 0.001, epsilon² = 0.1360
Physical_Activity: p < 0.001, epsilon² = 0.0894
Work_Hours: p < 0.001, epsilon² = 0.0575
Sleep_Duration: p < 0.001, epsilon² = 0.0318

Categorical hypotheses:
Meditation_Practice: p < 0.001, Cramer's V = 0.3040
Smoking_Habit: p < 0.001, Cramer's V = 0.2965
Marital_Status: p < 0.001, Cramer's V = 0.2064
Exercise_Type: p < 0.001, Cramer's V = 0.1995
Gender: p < 0.001, Cramer's V = 0.1460

Frozen test set used: False
